# 3.3 · 异常值检测 / Outlier Detection

> **课程定位 / Where this fits**
> **Part 3 第 3 课**。缺失值处理完，下一个脏数据问题：异常值。**核心哲学：异常值不是"删掉就好"——先分清是"错误"还是"真实的罕见"**，前者修正/删除，后者可能正是金矿（欺诈、故障）。
> Outliers aren't "just delete them" — first distinguish errors from genuine rarities. The latter (fraud, faults) may be the gold.

> 💡 **面试相关 / Interview-relevant**
> - "怎么检测异常值" ★★★★（多方法 + 何时用哪个）
> - "为什么 z-score 对异常值不稳健" ★★★★（异常值抬高自己的门槛）
> - "Isolation Forest 原理" ★★★
> - "单变量 vs 多变量异常" ★★★★（马氏距离）

---

## 学习目标 / Learning Objectives
1. 区分**单变量 vs 多变量**异常（有的点每个维度都正常，组合起来异常）。
2. 掌握 4 类方法：**统计(z/IQR) / 距离(马氏) / 密度(LOF) / 树(Isolation Forest)**。
3. 理解 **z-score 的自我掩盖问题**，改用稳健版（MAD）。
4. 形成"检测后**怎么处置**"的决策框架（删 / 截断 / 保留 / 单独建模）。

## 目录 / TOC
1. [异常值的三种身份 ⭐](#1)
2. [🏠 数据 + 注入异常](#2)
3. [z-score 与它的自我掩盖 ⚠](#3)
4. [IQR / 箱线图法](#4)
5. [马氏距离：多变量异常 ⭐](#5)
6. [Isolation Forest ⭐](#6)
7. [LOF：局部密度](#7)
8. [方法横向对比](#8)
9. [⚠ 检测之后怎么处置](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 异常值的三种身份 ⭐ / Three Kinds of Outliers

**删之前先问：它是什么？**

| 身份 | 例子 | 处置 |
|---|---|---|
| **数据错误** | 年龄 = 999、价格 = -50 | 修正或删除 |
| **真实罕见值** | 亿万富翁的收入、双胞胎 | **保留**（删了就偏）|
| **关注目标本身** | 欺诈交易、设备故障、罕见病 | **不是噪声, 是信号** — 整个异常检测领域为它而生 |

| 维度视角 | 含义 |
|---|---|
| **单变量 (univariate)** | 单看一列就异常（年龄 200）|
| **多变量 (multivariate)** | 每列都正常, **组合**异常（身高 150cm + 体重 120kg）⭐ |

> 💡 多变量异常是新手最容易漏的——逐列看箱线图全正常，但马氏距离/Isolation Forest 能抓住。
> Multivariate outliers are normal on every axis but anomalous in combination — column-wise boxplots miss them.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 演示多变量异常: 两个相关变量, 注入"每维正常但组合异常"的点
# Multivariate outlier: normal on each axis, anomalous together
mean = [0, 0]; cov = [[1, 0.85], [0.85, 1]]      # 强正相关 / strong correlation
normal_pts = rng.multivariate_normal(mean, cov, 500)
mv_outlier = np.array([[2.5, -2.5]])             # x,y 各自不极端, 但违反相关结构

pts = np.vstack([normal_pts, mv_outlier])
plt.figure(figsize=(5.5, 5.5))
plt.scatter(normal_pts[:,0], normal_pts[:,1], s=12, alpha=0.5, label="normal")
plt.scatter(*mv_outlier.T, s=200, c="red", marker="*", label="多变量异常", zorder=5)
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.legend(); plt.title("红星: x∈[-3,3]正常, y∈[-3,3]正常\n但 (2.5,-2.5) 违反正相关 → 多变量异常")
plt.tight_layout(); plt.show()
print(f"红星点 x={mv_outlier[0,0]} (|z|={abs(mv_outlier[0,0]):.1f}, 不极端)")
print(f"       y={mv_outlier[0,1]} (|z|={abs(mv_outlier[0,1]):.1f}, 不极端)")
print("逐列 z-score 都抓不到! 需要马氏距离 (第5节)")


<a id="2"></a>
## 2. 🏠 数据 + 注入异常 / Data + Injected Outliers

用 California Housing（0.2 节见过），注入几个已知异常，看各方法能不能抓回来。


In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame[["MedInc","HouseAge","AveRooms","Population"]].copy()
print(f"原始 shape: {df.shape}")
print(df.describe().round(2))
# 真实数据里 AveRooms 已有极端值 (某街区平均 100+ 房间, 数据错误) / data already has errors
print(f"\nAveRooms 最大值 = {df.AveRooms.max():.0f} (平均一户 100+ 房间?? 数据错误)")


<a id="3"></a>
## 3. z-score 与它的自我掩盖 ⚠ / z-score & Self-masking

$$z_i = \frac{x_i - \bar{x}}{s}, \qquad |z_i| > 3 \text{ 判为异常}$$

**致命缺陷（2.1 节预告过）**：均值 $\bar{x}$ 和标准差 $s$ **本身被异常值污染** → 异常值**抬高了判定自己的门槛** = 自我掩盖（masking）。
The mean and std are themselves corrupted by the outliers — the outlier raises the very threshold meant to catch it.

**稳健版**：用中位数 + MAD 替代：
$$z^{\text{robust}}_i = \frac{x_i - \text{median}(x)}{1.4826 \cdot \text{MAD}}$$


In [ ]:
x = df["MedInc"].values

# 普通 z-score / classic z-score
z = (x - x.mean()) / x.std()
n_classic = (np.abs(z) > 3).sum()

# 稳健 z-score (中位数 + MAD) / robust z-score
med = np.median(x)
mad = np.median(np.abs(x - med))
z_robust = (x - med) / (1.4826 * mad)
n_robust = (np.abs(z_robust) > 3).sum()

print(f"普通 z-score (|z|>3): 抓到 {n_classic} 个异常")
print(f"稳健 z-score (|z|>3): 抓到 {n_robust} 个异常")
print(f"\n演示自我掩盖: 人为加 5 个超大值")
x_dirty = np.append(x, [50, 60, 70, 80, 90])
z_dirty = (x_dirty - x_dirty.mean()) / x_dirty.std()
print(f"  加污染后, 这 5 个点的 |z| = {np.abs(z_dirty[-5:]).round(1)}")
print(f"  其中 |z|>3 的只有 {(np.abs(z_dirty[-5:])>3).sum()} 个 — 它们互相抬高了 std, 掩盖了彼此!")
z_rob_dirty = (x_dirty - np.median(x_dirty)) / (1.4826*np.median(np.abs(x_dirty-np.median(x_dirty))))
print(f"  稳健版全部抓到: |z_robust|>3 有 {(np.abs(z_rob_dirty[-5:])>3).sum()}/5 ✓")


<a id="4"></a>
## 4. IQR / 箱线图法 / IQR Method

$$\text{IQR} = Q_3 - Q_1, \quad \text{异常} = \big[Q_1 - 1.5\,\text{IQR},\; Q_3 + 1.5\,\text{IQR}\big] \text{ 之外}$$

**天生稳健**（分位数不受极值影响），无需正态假设。箱线图的"须外点"就是它。1.5 倍是 Tukey 的惯例（3 倍 = "极端异常"）。


In [ ]:
def iqr_outliers(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (x < lo) | (x > hi), (lo, hi)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, df.columns):
    mask, (lo, hi) = iqr_outliers(df[col].values)
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f"{col}\n{mask.sum()} outliers ({mask.mean():.1%})")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 马氏距离：多变量异常 ⭐ / Mahalanobis Distance

z-score 的多变量推广。**它考虑协方差** → 能抓住第 1 节那种"违反相关结构"的点。

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$

**直觉**：$\boldsymbol{\Sigma}^{-1}$ 把数据"白化"——沿相关方向拉伸的椭圆变回正圆，再量欧氏距离。**正态下 $D_M^2 \sim \chi^2_d$** → 可定阈值。
$\Sigma^{-1}$ whitens the correlated ellipse back to a circle; under normality $D_M^2$ follows chi-square with d df.


In [ ]:
from scipy.stats import chi2

# 用第 1 节的二维相关数据 / use the 2D correlated data
mu = normal_pts.mean(axis=0)
Sigma = np.cov(normal_pts.T)
Sigma_inv = np.linalg.inv(Sigma)

def mahalanobis(X, mu, Sigma_inv):
    diff = X - mu
    return np.sqrt(np.sum(diff @ Sigma_inv * diff, axis=1))

dm_normal = mahalanobis(normal_pts, mu, Sigma_inv)
dm_outlier = mahalanobis(mv_outlier, mu, Sigma_inv)[0]

# 阈值: chi2(df=2) 的 97.5% 分位 / threshold from chi-square
thresh = np.sqrt(chi2.ppf(0.975, df=2))
print(f"多变量异常点的马氏距离 = {dm_outlier:.2f}")
print(f"它的欧氏距离 = {np.sqrt((mv_outlier[0]**2).sum()):.2f} (不算远!)")
print(f"chi2 阈值 (97.5%) = {thresh:.2f}")
print(f"→ 马氏距离 {dm_outlier:.1f} >> 阈值 {thresh:.1f}, 成功抓到!")
print(f"  而 z-score 看每维都 < 3, 完全漏掉 — 这就是马氏距离的价值")


<a id="6"></a>
## 6. Isolation Forest ⭐ / 隔离森林

**思路天才地简单**：随机切分特征空间。**异常点更容易被孤立**（少数几刀就能把它单独切出来），正常点埋在密集区需要很多刀。

$$\text{异常分数} \propto \frac{1}{\text{平均隔离所需切分次数}}$$

**优点**：无需距离/密度计算 → **高维快**、可扩展、无分布假设。**工业界异常检测首选之一**。
No distance/density computation — fast in high dimensions, scales well, no distributional assumptions. An industry default.


In [ ]:
from sklearn.ensemble import IsolationForest

X = df[["MedInc","HouseAge","AveRooms","Population"]].values
iso = IsolationForest(contamination=0.02, random_state=0)  # 预期 2% 异常 / expected 2%
labels = iso.fit_predict(X)        # -1 = 异常, 1 = 正常
scores = iso.score_samples(X)      # 越小越异常

n_anom = (labels == -1).sum()
print(f"Isolation Forest 标记 {n_anom} 个异常 ({n_anom/len(X):.1%})")
print(f"\n最异常的 5 个街区:")
df_scored = df.assign(anomaly_score=scores).sort_values("anomaly_score")
print(df_scored.head(5).round(1))
print("\n看这些点: AveRooms 极大 (数据错误) 或 Population 极端 — 合理")


<a id="7"></a>
## 7. LOF：局部密度 / Local Outlier Factor

**核心创新：局部性**。一个点"异常"与否，看它的密度**相对于邻居的密度**——而不是全局。

**为什么重要**：数据有多个不同密度的簇时，全局方法会把"稀疏簇"整个误判为异常。LOF 只问"你比你的邻居稀疏吗"。
When data has clusters of differing densities, global methods flag the whole sparse cluster; LOF asks only "are you sparser than YOUR neighbors".

$$\text{LOF}(p) = \frac{\text{p 的邻居们的平均局部密度}}{\text{p 自己的局部密度}}, \qquad \text{LOF} \gg 1 \Rightarrow \text{异常}$$


In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# LOF 对尺度敏感 → 先标准化 / scale-sensitive, standardize first
Xs = StandardScaler().fit_transform(X)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_labels = lof.fit_predict(Xs)
lof_scores = -lof.negative_outlier_factor_   # 越大越异常

print(f"LOF 标记 {(lof_labels==-1).sum()} 个异常")
# 对比 IF 和 LOF 的重叠 / overlap between IF and LOF
overlap = ((labels==-1) & (lof_labels==-1)).sum()
print(f"IF 与 LOF 都标记的: {overlap} 个 (方法不同, 结果部分重叠是正常的)")
print("\nLOF 适合: 多密度簇数据; IF 适合: 高维大数据; 实践常两者都跑取交集/并集")


<a id="8"></a>
## 8. 方法横向对比 / Method Comparison

| 方法 | 类型 | 多变量 | 稳健 | 高维 | 假设 |
|---|---|---|---|---|---|
| z-score | 统计 | ❌ | ❌ 自我掩盖 | — | 近正态 |
| 稳健 z (MAD) | 统计 | ❌ | ✅ | — | 单峰 |
| IQR | 统计 | ❌ | ✅ | — | 无 |
| 马氏距离 | 距离 | ✅ | ❌ (用普通协方差) | 中 | 近正态椭圆 |
| **Isolation Forest** | 树 | ✅ | ✅ | ✅ 强 | 无 |
| LOF | 密度 | ✅ | ✅ | 弱(维度诅咒) | 无 |

**选择口诀**：
- 单列快速筛 → IQR
- 怀疑相关结构异常 → 马氏（小维度）
- 高维 / 大数据 / 通用 → **Isolation Forest**
- 多密度簇 → LOF


<a id="9"></a>
## 9. ⚠ 检测之后怎么处置 / What to Do After Detection

**检测只是第一步, 处置才是关键**——回到第 1 节的"三种身份":

| 处置 | 何时 | 风险 |
|---|---|---|
| **删除** | 确认是数据错误 + MCAR | 删真实罕见值 = 引入偏差 |
| **截断 (winsorize)** | 想保留行但削峰 | 改变了真实分布 |
| **变换 (log)** | 右偏长尾 (fare/收入) | 异常值被自然压缩, 常比删除好 ⭐ |
| **保留 + 稳健模型** | 异常是真实的 | 用树模型/稳健回归(Part 4.15) |
| **单独建模** | 异常就是目标 (欺诈) | 这是 Part 4.12/6 异常检测的正题 |

> 💡 **最常见的错误**: 无脑删除所有"异常值"。删之前必须问"这是错误还是真实"。删掉真实的尾部 = 让模型对现实视而不见。
> The most common mistake: blindly deleting all "outliers". Deleting genuine tails blinds the model to reality.


In [ ]:
# 演示 winsorize (截断) vs log 变换 / winsorize vs log transform
from scipy.stats.mstats import winsorize
fare_like = df["MedInc"].values

wins = winsorize(fare_like, limits=[0, 0.02])    # 顶部 2% 截断到 98 分位
logged = np.log1p(fare_like)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(fare_like, bins=50); axes[0].set_title(f"原始 (skew={st.skew(fare_like):.2f})")
axes[1].hist(wins, bins=50); axes[1].set_title(f"winsorize 2% (skew={st.skew(wins):.2f})")
axes[2].hist(logged, bins=50); axes[2].set_title(f"log1p (skew={st.skew(logged):.2f})")
plt.tight_layout(); plt.show()
print("log 变换常是处理右偏'异常'的最优雅方案 — 不删数据, 自然压缩长尾")


<a id="10"></a>
## 10. 小结 / Summary

```
先问身份: 数据错误(删/修) / 真实罕见(留) / 目标本身(异常检测)
维度: 单变量(逐列) vs 多变量(组合异常, z-score 漏)

方法谱:
  z-score — 不稳健(自我掩盖) → 用 MAD 稳健版
  IQR     — 稳健, 单列首选
  马氏距离 — 多变量, 抓相关结构异常 (D²~χ²)
  Isolation Forest — 高维/大数据/通用首选 ⭐
  LOF     — 多密度簇, 局部视角

处置 ≠ 删除: winsorize / log变换 / 稳健模型 / 单独建模
```

### 💡 面试速查
1. **z-score 自我掩盖**：异常值抬高自己的判定门槛 → 用 MAD
2. **多变量异常**：每维正常组合异常 → 马氏 / IF
3. **Isolation Forest**：随机切分, 异常易孤立, 高维快
4. **检测后先问身份再处置**, 别无脑删

### 下一节
**3.4 特征缩放**——KNN/LOF/马氏都对尺度敏感（本课一直在 `StandardScaler` 后才用）。下一课系统讲标准化/归一化/稳健缩放。
